In [ ]:
# =========================================
# 🧩 Deepfake Detection with EfficientNet, SE, and Attention Modules
# Dataset: WildDeepfake (KaggleHub)
# =========================================

# Install required packages first
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ Successfully installed {package}")
    except subprocess.CalledProcessError:
        print(f"❌ Failed to install {package}")

# Install kagglehub if not available
try:
    import kagglehub
    print("✅ kagglehub already installed")
except ImportError:
    print("📦 Installing kagglehub...")
    install_package("kagglehub")
    import kagglehub

# Standard imports
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Multiply, Reshape, Conv2D, Add, Lambda, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print("✅ All imports successful!")

# Set memory growth for GPU if available
if tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(tf.config.list_physical_devices('GPU')[0], True)
    print("🎮 GPU memory growth enabled")
else:
    print("💻 Running on CPU")

# =========================================
# 1️⃣ Download and Setup Dataset
# =========================================

def download_and_setup_dataset():
    """Download and setup the WildDeepfake dataset with proper error handling"""
    try:
        print("⬇️ Downloading WildDeepfake dataset...")
        download_path = kagglehub.dataset_download("maysuni/wild-deepfake")
        print(f"✅ Dataset downloaded at: {download_path}")
        
        # Unzip if necessary
        dataset_path = download_path
        for file in os.listdir(download_path):
            if file.endswith(".zip"):
                zip_path = os.path.join(download_path, file)
                extract_dir = os.path.join(download_path, "WildDeepfake")
                if not os.path.exists(extract_dir):
                    print("📦 Extracting dataset...")
                    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                        zip_ref.extractall(extract_dir)
                    print("✅ Dataset extracted successfully")
                dataset_path = extract_dir
                break

        # Verify the path and find the correct directory containing 'fake' and 'real' folders
        if not os.path.exists(os.path.join(dataset_path, "fake")) or not os.path.exists(os.path.join(dataset_path, "real")):
            print("🔍 Searching for 'fake' and 'real' directories...")
            found_path = None
            for root, dirs, files in os.walk(download_path):
                if 'fake' in dirs and 'real' in dirs:
                    found_path = root
                    break
            if found_path:
                dataset_path = found_path
                print(f"✅ Found directories at: {dataset_path}")
            else:
                raise FileNotFoundError("❌ Could not find 'fake' and 'real' directories in dataset.")

        print(f"📁 Using dataset path: {dataset_path}")
        
        # Verify dataset contents
        fake_dir = os.path.join(dataset_path, "fake")
        real_dir = os.path.join(dataset_path, "real")
        
        if os.path.exists(fake_dir) and os.path.exists(real_dir):
            fake_count = len([f for f in os.listdir(fake_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            real_count = len([f for f in os.listdir(real_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            print(f"📊 Dataset verification:")
            print(f"   - Fake images: {fake_count}")
            print(f"   - Real images: {real_count}")
            print(f"   - Total images: {fake_count + real_count}")
        
        return dataset_path
        
    except Exception as e:
        print(f"❌ Error downloading dataset: {str(e)}")
        print("💡 Alternative: Please manually download the dataset or check your Kaggle API configuration")
        
        # Provide alternative instructions
        print("\n🔧 Alternative setup instructions:")
        print("1. Go to https://www.kaggle.com/datasets/maysuni/wild-deepfake")
        print("2. Download the dataset manually")
        print("3. Extract it to a folder with 'fake' and 'real' subdirectories")
        print("4. Update the dataset_path variable below")
        
        # Return a placeholder path - user should update this
        return r"C:\path\to\your\dataset"  # User should update this path

# Download and setup dataset
dataset_path = download_and_setup_dataset()

# =========================================
# 2️⃣ Data Preprocessing and Augmentation
# =========================================

def setup_data_generators(dataset_path):
    """Setup data generators with error handling"""
    try:
        img_size = (224, 224)
        batch_size = 32

        # Enhanced data augmentation for training
        train_datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=20,
            width_shift_range=0.15,
            height_shift_range=0.15,
            shear_range=0.15,
            zoom_range=0.15,
            brightness_range=[0.8, 1.2],
            horizontal_flip=True,
            fill_mode='nearest',
            validation_split=0.2
        )

        # Simple rescaling for validation
        val_datagen = ImageDataGenerator(
            rescale=1./255, 
            validation_split=0.2
        )

        # Create training generator
        train_gen = train_datagen.flow_from_directory(
            dataset_path,
            target_size=img_size,
            batch_size=batch_size,
            class_mode='binary',
            subset='training',
            shuffle=True,
            seed=42
        )

        # Create validation generator
        val_gen = val_datagen.flow_from_directory(
            dataset_path,
            target_size=img_size,
            batch_size=batch_size,
            class_mode='binary',
            subset='validation',
            shuffle=False,  # Fixed: was 'f' instead of False
            seed=42
        )

        print(f"✅ Data generators created successfully:")
        print(f"   - Training samples: {train_gen.samples}")
        print(f"   - Validation samples: {val_gen.samples}")
        print(f"   - Class indices: {train_gen.class_indices}")
        
        return train_gen, val_gen, img_size, batch_size
        
    except Exception as e:
        print(f"❌ Error setting up data generators: {str(e)}")
        print("💡 Please check if the dataset path is correct and contains 'fake' and 'real' folders")
        return None, None, None, None

# Setup data generators
train_gen, val_gen, img_size, batch_size = setup_data_generators(dataset_path)

# Verify generators were created successfully
if train_gen is None or val_gen is None:
    print("❌ Failed to create data generators. Please check your dataset path.")
else:
    print("✅ Data preprocessing setup complete!")

✅ kagglehub already installed
✅ All imports successful!
💻 Running on CPU
⬇️ Downloading WildDeepfake dataset...


  6%|▌         | 637M/10.1G [32:52<8:23:11, 339kB/s]  

❌ Error downloading dataset: ('Connection broken: IncompleteRead(668606644 bytes read, 10222342170 more expected)', IncompleteRead(668606644 bytes read, 10222342170 more expected))
💡 Alternative: Please manually download the dataset or check your Kaggle API configuration

🔧 Alternative setup instructions:
1. Go to https://www.kaggle.com/datasets/maysuni/wild-deepfake
2. Download the dataset manually
3. Extract it to a folder with 'fake' and 'real' subdirectories
4. Update the dataset_path variable below
❌ Error setting up data generators: [WinError 3] The system cannot find the path specified: 'C:\\path\\to\\your\\dataset'
💡 Please check if the dataset path is correct and contains 'fake' and 'real' folders
❌ Failed to create data generators. Please check your dataset path.


In [ ]:
# =========================================
# 🔧 DIAGNOSTIC AND FIXES
# =========================================

# First, let's check your data generators and fix potential issues
print("🔍 Diagnosing Training Issues...")

# 1. Check data generator setup
print(f"Training samples: {train_gen.samples if train_gen else 'None'}")
print(f"Validation samples: {val_gen.samples if val_gen else 'None'}")
print(f"Class indices: {train_gen.class_indices if train_gen else 'None'}")

# 2. Check a few batches to ensure data is loading correctly
if train_gen:
    print("\n📊 Checking data distribution...")
    x_batch, y_batch = next(train_gen)
    print(f"Batch shape: {x_batch.shape}")
    print(f"Labels shape: {y_batch.shape}")
    print(f"Label distribution in batch: {np.bincount(y_batch.astype(int))}")
    print(f"Input range: [{x_batch.min():.3f}, {x_batch.max():.3f}]")
    
    # Check for class imbalance
    try:
        fake_count = len([f for f in os.listdir(os.path.join(dataset_path, "fake")) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        real_count = len([f for f in os.listdir(os.path.join(dataset_path, "real")) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        imbalance_ratio = max(fake_count, real_count) / min(fake_count, real_count)
        print(f"Class imbalance ratio: {imbalance_ratio:.2f}")
        
        if imbalance_ratio > 2.0:
            print("⚠️ Significant class imbalance detected!")
    except Exception as e:
        print(f"Could not check class balance: {e}")

    # 3. Visualize a few samples
    print("\n🖼️ Sample images from batch:")
    plt.figure(figsize=(12, 8))
    for i in range(min(8, len(x_batch))):
        plt.subplot(2, 4, i+1)
        plt.imshow(x_batch[i])
        plt.title(f"Label: {'Fake' if y_batch[i] > 0.5 else 'Real'}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # 4. Check if dataset path is actually valid
    print(f"\n📁 Dataset path verification:")
    print(f"Dataset path: {dataset_path}")
    print(f"Path exists: {os.path.exists(dataset_path)}")
    if os.path.exists(dataset_path):
        print(f"Contents: {os.listdir(dataset_path)}")
        fake_dir = os.path.join(dataset_path, "fake")
        real_dir = os.path.join(dataset_path, "real")
        print(f"Fake dir exists: {os.path.exists(fake_dir)}")
        print(f"Real dir exists: {os.path.exists(real_dir)}")
        
        if os.path.exists(fake_dir):
            print(f"Sample fake files: {os.listdir(fake_dir)[:5]}")
        if os.path.exists(real_dir):
            print(f"Sample real files: {os.listdir(real_dir)[:5]}")

print("\n✅ Diagnostic check completed!")

# Reset the generator to start from beginning
if train_gen:
    train_gen.reset()
if val_gen:
    val_gen.reset()

In [ ]:
# =========================================
# 🚀 IMPROVED MODEL CONFIGURATIONS (NETWORK-RESILIENT)
# =========================================

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

def build_improved_baseline_offline_safe():
    """Improved baseline with offline-safe weight loading"""
    try:
        # Try to load with imagenet weights first
        print("🌐 Attempting to download ImageNet weights...")
        base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        print("✅ ImageNet weights loaded successfully!")
    except Exception as e:
        print(f"❌ Failed to download ImageNet weights: {str(e)}")
        print("🔄 Using random initialization instead...")
        # Fallback to random weights if download fails
        base = EfficientNetB0(weights=None, include_top=False, input_shape=(224, 224, 3))
        print("✅ Model created with random weights")

    # Unfreeze more layers for better learning
    for layer in base.layers[:-40]:  # Unfreeze more layers
        layer.trainable = False

    x = base.output
    x = GlobalAveragePooling2D(name="baseline_gap")(x)
    x = Dropout(0.5, name="baseline_dropout1")(x)  # Increased dropout
    x = Dense(256, activation='relu', name="baseline_dense1")(x)  # Reduced neurons
    x = Dropout(0.3, name="baseline_dropout2")(x)
    x = Dense(128, activation='relu', name="baseline_dense2")(x)  # Added layer
    x = Dropout(0.2, name="baseline_dropout3")(x)
    output = Dense(1, activation='sigmoid', name="baseline_output")(x)

    model = Model(inputs=base.input, outputs=output, name="M1_Improved_Baseline")

    # Better optimizer settings
    model.compile(
        optimizer=Adam(learning_rate=2e-4, beta_1=0.9, beta_2=0.999),  # Higher initial LR
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )
    return model

def build_simple_cnn_alternative():
    """Alternative simple CNN model if EfficientNet fails completely"""
    from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Flatten

    print("🏗️ Building simple CNN as alternative...")

    inputs = tf.keras.Input(shape=(224, 224, 3))

    # Block 1
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)

    # Block 2
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)

    # Block 3
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)

    # Block 4
    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling2D()(x)

    # Classifier
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=output, name="Simple_CNN_Alternative")
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )

    return model

def get_improved_callbacks(model_name):
    """Improved callbacks with better patience"""
    return [
        ModelCheckpoint(
            f"{model_name}.keras",  # Use new format
            save_best_only=True,
            monitor='val_accuracy',
            mode='max',
            verbose=1,
            # save_format='keras' # Removed this argument
        ),
        EarlyStopping(
            patience=10,  # Increased patience
            restore_best_weights=True,
            monitor='val_accuracy',
            verbose=1,
            min_delta=0.001  # Minimum improvement threshold
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,  # More aggressive reduction
            patience=5,  # Increased patience
            min_lr=1e-7,
            verbose=1,
            cooldown=2
        )
    ]

# Check network connectivity first
def check_network_connectivity():
    """Check if we can access the internet"""
    import urllib.request
    try:
        urllib.request.urlopen('https://www.google.com', timeout=5)
        return True
    except:
        return False

# Network check and model building
print("🔍 Checking network connectivity...")
network_available = check_network_connectivity()

if network_available:
    print("✅ Network available, attempting EfficientNet download...")
    try:
        improved_model = build_improved_baseline_offline_safe()
        model_type = "EfficientNet"
    except Exception as e:
        print(f"❌ EfficientNet failed even with network: {str(e)}")
        print("🔄 Falling back to simple CNN...")
        improved_model = build_simple_cnn_alternative()
        model_type = "Simple CNN"
else:
    print("❌ No network connectivity detected")
    print("🔄 Using simple CNN alternative...")
    improved_model = build_simple_cnn_alternative()
    model_type = "Simple CNN (Offline)"

print(f"✅ {model_type} model created successfully")
print(f"Total parameters: {improved_model.count_params():,}")
print(f"Trainable parameters: {sum([tf.keras.backend.count_params(w) for w in improved_model.trainable_weights]):,}")

# Test the model with a reduced number of epochs
if train_gen is not None and val_gen is not None:
    print(f"\n🧪 Testing {model_type} model for 3 epochs...")
    test_callbacks = get_improved_callbacks("test_improved_baseline")

    try:
        test_history = improved_model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=3,
            callbacks=test_callbacks,
            verbose=1
        )

        print(f"Test results after 3 epochs:")
        print(f"Training accuracy: {test_history.history['accuracy'][-1]:.4f}")
        print(f"Validation accuracy: {test_history.history['val_accuracy'][-1]:.4f}")

        if test_history.history['val_accuracy'][-1] > 0.55:
            print("✅ Model is working well!")
        else:
            print("⚠️ Model performance could be improved with more training or better data.")

    except Exception as e:
        print(f"❌ Error during training: {str(e)}")
        print("💡 This might be due to data generator issues. Please check your dataset.")
else:
    print("⚠️ Data generators not available. Please run the data setup cells first.")

In [ ]:
# =========================================
# 🚀 IMPROVED MODEL CONFIGURATIONS
# =========================================

def build_improved_baseline():
    """Improved baseline with better hyperparameters"""
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # Unfreeze more layers for better learning
    for layer in base.layers[:-40]:  # Unfreeze more layers
        layer.trainable = False
    
    x = base.output
    x = GlobalAveragePooling2D(name="baseline_gap")(x)
    x = Dropout(0.5, name="baseline_dropout1")(x)  # Increased dropout
    x = Dense(256, activation='relu', name="baseline_dense1")(x)  # Reduced neurons
    x = Dropout(0.3, name="baseline_dropout2")(x)
    x = Dense(128, activation='relu', name="baseline_dense2")(x)  # Added layer
    x = Dropout(0.2, name="baseline_dropout3")(x)
    output = Dense(1, activation='sigmoid', name="baseline_output")(x)
    
    model = Model(inputs=base.input, outputs=output, name="M1_Improved_Baseline")
    
    # Better optimizer settings
    model.compile(
        optimizer=Adam(learning_rate=2e-4, beta_1=0.9, beta_2=0.999),  # Higher initial LR
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )
    return model


def get_improved_callbacks(model_name):
    """Improved callbacks with better patience"""
    return [
        ModelCheckpoint(
            f"{model_name}.keras",  # Use new format
            save_best_only=True, 
            monitor='val_accuracy', 
            mode='max',
            verbose=1,
            save_format='keras'
        ),
        EarlyStopping(
            patience=10,  # Increased patience
            restore_best_weights=True,
            monitor='val_accuracy',
            verbose=1,
            min_delta=0.001  # Minimum improvement threshold
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,  # More aggressive reduction
            patience=5,  # Increased patience
            min_lr=1e-7,
            verbose=1,
            cooldown=2
        )
    ]


# Build and test improved model
print("🏗️ Building improved baseline model...")
try:
    improved_model = build_improved_baseline()
    print(f"✅ Improved baseline created")
    print(f"Total parameters: {improved_model.count_params():,}")
    print(f"Trainable parameters: {sum([tf.keras.backend.count_params(w) for w in improved_model.trainable_weights]):,}")
    
    # Quick test training for 3 epochs
    print("\n🧪 Testing improved model for 3 epochs...")
    test_callbacks = get_improved_callbacks("test_improved_baseline")
    
    test_history = improved_model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=3,
        callbacks=test_callbacks,
        verbose=1
    )
    
    print(f"Test results after 3 epochs:")
    print(f"Training accuracy: {test_history.history['accuracy'][-1]:.4f}")
    print(f"Validation accuracy: {test_history.history['val_accuracy'][-1]:.4f}")
    
    if test_history.history['val_accuracy'][-1] > 0.55:
        print("✅ Improvement detected! The fixed model is working better.")
    else:
        print("⚠️ Still having issues. Let's check data quality...")
        
except Exception as e:
    print(f"❌ Error with improved model: {str(e)}")

In [ ]:
# =========================================
# 4️⃣ Model 1: Baseline EfficientNet
# =========================================

def build_baseline_efficientnet():
    """Baseline EfficientNet without any attention modules"""
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    # Freeze first few layers
    for layer in base.layers[:-20]:
        layer.trainable = False

    x = base.output
    x = GlobalAveragePooling2D(name="baseline_gap")(x)
    x = Dropout(0.3, name="baseline_dropout1")(x)
    x = Dense(512, activation='relu', name="baseline_dense1")(x)
    x = Dropout(0.2, name="baseline_dropout2")(x)
    output = Dense(1, activation='sigmoid', name="baseline_output")(x)

    model = Model(inputs=base.input, outputs=output, name="M1_Baseline_EfficientNet")
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build and summarize model
model_1 = build_baseline_efficientnet()
print("🏗️ Model 1: Baseline EfficientNet")
print(f"Total parameters: {model_1.count_params():,}")
model_1.summary()

# =========================================
# 5️⃣ Model 2: EfficientNet + Squeeze-and-Excitation
# =========================================

def build_efficientnet_se():
    """EfficientNet with Squeeze-and-Excitation enhancement"""
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    # Freeze first few layers
    for layer in base.layers[:-20]:
        layer.trainable = False

    x = base.output
    x = squeeze_excite_block(x, ratio=16, name_prefix="enhanced_se")
    x = GlobalAveragePooling2D(name="se_gap")(x)
    x = Dropout(0.3, name="se_dropout1")(x)
    x = Dense(512, activation='relu', name="se_dense1")(x)
    x = Dropout(0.2, name="se_dropout2")(x)
    output = Dense(1, activation='sigmoid', name="se_output")(x)

    model = Model(inputs=base.input, outputs=output, name="M2_EfficientNet_SE")
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build and summarize model
model_2 = build_efficientnet_se()
print("🏗️ Model 2: EfficientNet + SE")
print(f"Total parameters: {model_2.count_params():,}")
model_2.summary()

# =========================================
# 6️⃣ Model 3: EfficientNet + CBAM Attention
# =========================================

def build_efficientnet_cbam():
    """EfficientNet with CBAM attention mechanism"""
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    # Freeze first few layers
    for layer in base.layers[:-20]:
        layer.trainable = False

    x = base.output
    x = cbam_attention_block(x, reduction_ratio=16, name_prefix="cbam_block") # Changed name_prefix here
    x = GlobalAveragePooling2D(name="model3_gap")(x)
    x = Dropout(0.3, name="model3_dropout1")(x) # Changed name here
    x = Dense(512, activation='relu', name="model3_dense1")(x) # Changed name here
    x = Dropout(0.2, name="model3_dropout2")(x) # Changed name here
    output = Dense(1, activation='sigmoid', name="model3_output")(x) # Changed name here

    model = Model(inputs=base.input, outputs=output, name="M3_EfficientNet_CBAM")
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build and summarize model
model_3 = build_efficientnet_cbam()
print("🏗️ Model 3: EfficientNet + CBAM")
print(f"Total parameters: {model_3.count_params():,}")
model_3.summary()

In [ ]:
# =========================================
# 7️⃣ Model 4: EfficientNet + Self-Attention (FULLY FIXED)
# =========================================

def build_efficientnet_self_attention():
    """EfficientNet with Custom Self-Attention Layer"""
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # Freeze first few layers
    for layer in base.layers[:-20]:
        layer.trainable = False
    
    x = base.output
    # Use the custom self-attention layer
    x = self_attention_block(x, name_prefix="self_att")
    x = GlobalAveragePooling2D(name="self_att_gap")(x)
    x = Dropout(0.3, name="self_att_dropout1")(x)
    x = Dense(512, activation='relu', name="self_att_dense1")(x)
    x = Dropout(0.2, name="self_att_dropout2")(x)
    output = Dense(1, activation='sigmoid', name="self_att_output")(x)
    
    model = Model(inputs=base.input, outputs=output, name="M4_EfficientNet_SelfAttention")
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model


def build_efficientnet_multihead_attention():
    """EfficientNet with Keras MultiHeadAttention"""
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # Freeze first few layers
    for layer in base.layers[:-20]:
        layer.trainable = False
    
    x = base.output
    # Use multi-head attention
    x = multi_head_attention_block(x, num_heads=8, name_prefix="mha")
    x = GlobalAveragePooling2D(name="mha_gap")(x)
    x = Dropout(0.3, name="mha_dropout1")(x)
    x = Dense(512, activation='relu', name="mha_dense1")(x)
    x = Dropout(0.2, name="mha_dropout2")(x)
    output = Dense(1, activation='sigmoid', name="mha_output")(x)
    
    model = Model(inputs=base.input, outputs=output, name="M4_EfficientNet_MultiHeadAttention")
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model


def build_efficientnet_simplified_attention():
    """EfficientNet with Simplified Attention (most stable)"""
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # Freeze first few layers
    for layer in base.layers[:-20]:
        layer.trainable = False
    
    x = base.output
    # Use simplified attention block
    x = simplified_attention_block(x, name_prefix="simple_att")
    x = GlobalAveragePooling2D(name="simple_att_gap")(x)
    x = Dropout(0.3, name="simple_att_dropout1")(x)
    x = Dense(512, activation='relu', name="simple_att_dense1")(x)
    x = Dropout(0.2, name="simple_att_dropout2")(x)
    output = Dense(1, activation='sigmoid', name="simple_att_output")(x)
    
    model = Model(inputs=base.input, outputs=output, name="M4_EfficientNet_SimplifiedAttention")
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model


# Try building models in order of preference
print("🏗️ Building Model 4 with attention mechanism...")

try:
    model_4 = build_efficientnet_self_attention()
    print("✅ Model 4: EfficientNet + Custom Self-Attention")
    print(f"Total parameters: {model_4.count_params():,}")
    attention_type = "Custom Self-Attention"
except Exception as e:
    print(f"⚠️ Custom self-attention failed: {str(e)}")
    
    try:
        model_4 = build_efficientnet_multihead_attention()
        print("✅ Model 4: EfficientNet + Multi-Head Attention")
        print(f"Total parameters: {model_4.count_params():,}")
        attention_type = "Multi-Head Attention"
    except Exception as e:
        print(f"⚠️ Multi-head attention failed: {str(e)}")
        
        model_4 = build_efficientnet_simplified_attention()
        print("✅ Model 4: EfficientNet + Simplified Attention")
        print(f"Total parameters: {model_4.count_params():,}")
        attention_type = "Simplified Attention"

print(f"🎯 Using {attention_type} for Model 4")
model_4.summary()

In [ ]:
# =========================================
# 8️⃣ Model 5: EfficientNet + SE + CBAM (Hybrid)
# =========================================

def build_efficientnet_se_cbam():
    """EfficientNet with both SE and CBAM attention"""
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # Freeze first few layers
    for layer in base.layers[:-20]:
        layer.trainable = False
    
    x = base.output
    x = squeeze_excite_block(x, ratio=16, name_prefix="hybrid_se")
    x = cbam_attention_block(x, reduction_ratio=16, name_prefix="hybrid_cbam")
    x = GlobalAveragePooling2D(name="hybrid_gap")(x)
    x = Dropout(0.3, name="hybrid_dropout1")(x)
    x = Dense(512, activation='relu', name="hybrid_dense1")(x)
    x = Dropout(0.2, name="hybrid_dropout2")(x)
    output = Dense(1, activation='sigmoid', name="hybrid_output")(x)
    
    model = Model(inputs=base.input, outputs=output, name="M5_EfficientNet_SE_CBAM")
    model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build and summarize model
model_5 = build_efficientnet_se_cbam()
print("🏗️ Model 5: EfficientNet + SE + CBAM (Hybrid)")
print(f"Total parameters: {model_5.count_params():,}")
model_5.summary()

In [ ]:
# =========================================
# 9️⃣ Training Configuration and Utilities
# =========================================

def get_callbacks(model_name):
    """Get training callbacks for each model"""
    callbacks = [
        ModelCheckpoint(
            f"{model_name}.h5", 
            save_best_only=True, 
            monitor='val_accuracy', 
            mode='max',
            verbose=1
        ),
        EarlyStopping(
            patience=7, 
            restore_best_weights=True,
            monitor='val_accuracy',
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        )
    ]
    return callbacks


def train_model(model, model_name, epochs=25):
    """Train a single model with proper callbacks"""
    print(f"\n🚀 Training {model_name}")
    print("=" * 50)
    
    callbacks = get_callbacks(model_name)
    
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    
    return history

# Dictionary to store training histories
training_histories = {}

# Model configurations
models_to_train = [
    (model_1, "M1_Baseline_EfficientNet"),
    (model_2, "M2_EfficientNet_SE"),
    (model_3, "M3_EfficientNet_CBAM"),
    (model_4, "M4_EfficientNet_SelfAttention"),
    (model_5, "M5_EfficientNet_SE_CBAM")
]

# # =========================================
# # 5️⃣ Model Variants
# # =========================================
# models_config = [
#     ("M1_EfficientNet", False, False),
#     ("M2_EfficientNet_SE", True, False),
#     ("M3_EfficientNet_Att", False, True),
#     ("M4_SE_Att", True, True),
#     ("M5_EfficientNet_SE_Att", True, True)
# ]

# history_dict = {}


In [ ]:
# =========================================
# 🔟 Training All Models
# =========================================

for model, model_name in models_to_train:
    try:
        history = train_model(model, model_name, epochs=25)
        training_histories[model_name] = history.history
        print(f"✅ {model_name} training completed successfully!")
    except Exception as e:
        print(f"❌ Error training {model_name}: {str(e)}")
        continue

print("\n🎉 All model training completed!")

In [ ]:
# =========================================
# 1️⃣1️⃣ Enhanced Evaluation Function
# =========================================

def evaluate_model_comprehensive(model_path, model_name, generator):
    """Comprehensive model evaluation with metrics and visualizations"""
    
    # Custom objects for loading models with attention blocks
    custom_objects = {
        'squeeze_excite_block': squeeze_excite_block,
        'cbam_attention_block': cbam_attention_block,
        'self_attention_block': self_attention_block
    }
    
    try:
        model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)
    except:
        print(f"⚠️ Could not load {model_path}, skipping evaluation.")
        return None
    
    print(f"\n📊 Evaluating {model_name}")
    print("=" * 50)
    
    # Get predictions
    y_true = generator.classes
    y_pred = model.predict(generator, verbose=1).ravel()
    y_pred_classes = (y_pred > 0.5).astype(int)
    
    # Classification report
    print("\n📈 Classification Report:")
    print(classification_report(y_true, y_pred_classes, target_names=['Real', 'Fake']))
    
    # Confusion Matrix
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    cm = confusion_matrix(y_true, y_pred_classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    plt.title(f"{model_name} - Confusion Matrix")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    # ROC Curve
    plt.subplot(1, 2, 2)
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f"{model_name} - ROC Curve")
    plt.legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()
    
    # Calculate additional metrics
    accuracy = np.mean(y_true == y_pred_classes)
    precision = np.sum((y_pred_classes == 1) & (y_true == 1)) / np.sum(y_pred_classes == 1) if np.sum(y_pred_classes == 1) > 0 else 0
    recall = np.sum((y_pred_classes == 1) & (y_true == 1)) / np.sum(y_true == 1) if np.sum(y_true == 1) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'auc': roc_auc
    }
    
    print(f"\n📊 Summary Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"AUC: {roc_auc:.4f}")
    
    return metrics

In [ ]:
# =========================================
# 1️⃣2️⃣ Evaluate All Trained Models
# =========================================

model_results = {}

for _, model_name in models_to_train:
    model_path = f"{model_name}.h5"
    if os.path.exists(model_path):
        metrics = evaluate_model_comprehensive(model_path, model_name, val_gen)
        if metrics:
            model_results[model_name] = metrics
    else:
        print(f"⚠️ Model file {model_path} not found!")

# Create comparison DataFrame
import pandas as pd

if model_results:
    results_df = pd.DataFrame(model_results).T
    results_df = results_df.round(4)
    
    print("\n🏆 Model Comparison Results:")
    print("=" * 60)
    print(results_df)
    
    # Plot comparison
    plt.figure(figsize=(15, 10))
    
    # Accuracy comparison
    plt.subplot(2, 3, 1)
    plt.bar(results_df.index, results_df['accuracy'])
    plt.title('Model Accuracy Comparison')
    plt.xticks(rotation=45)
    plt.ylabel('Accuracy')
    
    # Precision comparison
    plt.subplot(2, 3, 2)
    plt.bar(results_df.index, results_df['precision'])
    plt.title('Model Precision Comparison')
    plt.xticks(rotation=45)
    plt.ylabel('Precision')
    
    # Recall comparison
    plt.subplot(2, 3, 3)
    plt.bar(results_df.index, results_df['recall'])
    plt.title('Model Recall Comparison')
    plt.xticks(rotation=45)
    plt.ylabel('Recall')
    
    # F1-Score comparison
    plt.subplot(2, 3, 4)
    plt.bar(results_df.index, results_df['f1_score'])
    plt.title('Model F1-Score Comparison')
    plt.xticks(rotation=45)
    plt.ylabel('F1-Score')
    
    # AUC comparison
    plt.subplot(2, 3, 5)
    plt.bar(results_df.index, results_df['auc'])
    plt.title('Model AUC Comparison')
    plt.xticks(rotation=45)
    plt.ylabel('AUC')
    
    # Overall performance radar chart
    plt.subplot(2, 3, 6)
    metrics_for_radar = results_df[['accuracy', 'precision', 'recall', 'f1_score', 'auc']].mean()
    plt.bar(metrics_for_radar.index, metrics_for_radar.values)
    plt.title('Average Performance Across All Models')
    plt.xticks(rotation=45)
    plt.ylabel('Score')
    
    plt.tight_layout()
    plt.show()
    
    # Find best model
    best_model = results_df['f1_score'].idxmax()
    print(f"\n🥇 Best performing model: {best_model}")
    print(f"F1-Score: {results_df.loc[best_model, 'f1_score']:.4f}")

print("\n✅ Comprehensive evaluation completed!")